# Rapprocher le référentiel et AquaMonitor complet

Ce notebook cherche les individus de AquaMonitor complet qui correspondent aux taxons
du fichier régional.

Les règles sont volontairement strictes :

- on ne crée jamais une nouvelle famille ;
- un genre déjà renseigné doit correspondre exactement ;
- si le genre régional est indéterminé, le dataset peut apporter plus de détail ;
- une classe peut être ajoutée, mais son chemin `REGNE → GENRE` doit déjà exister.

Pour AquaMonitor, un individu compte une seule fois, même s'il possède plusieurs images.


Le Parquet complet ne contient pas directement les rangs taxonomiques. On les
récupère avec `taxon_table.csv`, en reliant exactement `taxon_label` à `label`.

## 1. Imports et chemins

On retrouve automatiquement la racine du projet et les fichiers nécessaires.

In [ ]:
from pathlib import Path
import re
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REFERENCE_FILENAME = "classes_presentes_dans_taxons_experiments.xlsx"


def find_repo_root(start: Path | None = None) -> Path:
    """Locate the repository root containing the authoritative taxonomy file.

    Args:
        start: Optional directory from which to begin the upward search.

    Returns:
        Absolute path to the repository root.

    Raises:
        FileNotFoundError: If the reference workbook cannot be located.
    """
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / REFERENCE_FILENAME).exists():
            return candidate
    raise FileNotFoundError(f"Impossible de trouver data/{REFERENCE_FILENAME} depuis {current}.")


repo_root = find_repo_root()
reference_path = repo_root / "data" / REFERENCE_FILENAME
experiment_dir = repo_root / "experiments" / "aquamonitor"
am_path = experiment_dir / "data" / "metadata" / "aquamonitor.parquet.gzip"
reports_dir = experiment_dir / "reports" / "complet"
reports_dir.mkdir(parents=True, exist_ok=True)

print("Référence :", reference_path)
print("AM       :", am_path)
print("Rapports  :", reports_dir)

# La révision est épinglée : une mise à jour de la source est une décision explicite.
from huggingface_hub import hf_hub_download
DATASET_AM_ID = "mikkoim/aquamonitor"
DATASET_REVISION = "9a30834874d0428be529ab0d35d8aa3a687dd43a"
metadata_dir = experiment_dir / "data" / "metadata"
am_path = Path(hf_hub_download(
    repo_id=DATASET_AM_ID, repo_type="dataset", revision=DATASET_REVISION,
    filename="aquamonitor.parquet.gzip", local_dir=str(metadata_dir),
))
taxon_table_path = Path(hf_hub_download(
    repo_id=DATASET_AM_ID, repo_type="dataset", revision=DATASET_REVISION,
    filename="taxon_table.csv", local_dir=str(metadata_dir),
))

## 2. Charger le référentiel régional

On normalise seulement l'écriture des noms. On ne fait pas de rapprochement flou.

In [ ]:
TAXONOMY_RANKS = ["REGNE", "EMBRANCHEMENT", "CLASSE", "ORDRE", "FAMILLE", "GENRE"]
AM_RANKS = ["kingdom", "phylum", "class", "order", "family", "genus"]
SOURCE_CLASS_COLUMN = "Classe"
PRECISION_COLUMN = "Niveau_precision"
IMAGE_COUNT_COLUMN = "Nombre d'images"
REGION_COLUMN = "Fréquent en Bassin AG"
USEFUL_COLUMNS = [SOURCE_CLASS_COLUMN, *TAXONOMY_RANKS, REGION_COLUMN, PRECISION_COLUMN, IMAGE_COUNT_COLUMN]


def normalize_label(value) -> str | None:
    """Return a comparison key without changing taxonomic meaning.

    Only case, accents and separators are normalized; no fuzzy matching or
    synonym expansion is performed.
    """
    if pd.isna(value):
        return None
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(c for c in text if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]+", "_", text.casefold()).strip("_") or None


PLACEHOLDER_PATTERN = re.compile(r"^(sans |non |unknown|unidentified|nan$)", re.IGNORECASE)


def is_real_taxon(value) -> bool:
    """Return whether a cell contains a determined taxon rather than a placeholder."""
    return pd.notna(value) and not PLACEHOLDER_PATTERN.match(str(value).strip())

In [ ]:
taxonomy = pd.read_excel(reference_path, sheet_name="Sheet1", usecols=USEFUL_COLUMNS, engine="openpyxl")
for column in [SOURCE_CLASS_COLUMN, *TAXONOMY_RANKS]:
    taxonomy[column] = taxonomy[column].astype("string").str.strip().replace("", pd.NA)
taxonomy[PRECISION_COLUMN] = pd.to_numeric(taxonomy[PRECISION_COLUMN], errors="coerce").astype("Int64")
taxonomy[IMAGE_COUNT_COLUMN] = pd.to_numeric(taxonomy[IMAGE_COUNT_COLUMN], errors="coerce").fillna(0)
if (taxonomy[IMAGE_COUNT_COLUMN] < 0).any() or not np.allclose(taxonomy[IMAGE_COUNT_COLUMN] % 1, 0):
    raise ValueError("Le nombre d'images de référence doit être un entier positif.")
taxonomy[IMAGE_COUNT_COLUMN] = taxonomy[IMAGE_COUNT_COLUMN].astype("int64")
taxonomy["Frequent_bassin"] = taxonomy[REGION_COLUMN].astype("string").str.strip().str.casefold().eq("x")


def effective_level(row: pd.Series) -> int:
    """Resolve the authoritative rank used to identify a regional taxon.

    The declared precision is preferred. If absent, the deepest determined
    structured rank is used, excluding placeholders such as ``Sans Genre``.
    """
    if pd.notna(row[PRECISION_COLUMN]):
        return min(max(int(row[PRECISION_COLUMN]), 0), len(TAXONOMY_RANKS) - 1)
    real_levels = [i for i, rank in enumerate(TAXONOMY_RANKS) if is_real_taxon(row[rank])]
    if not real_levels:
        raise ValueError("Ligne régionale sans aucun taxon exploitable.")
    return max(real_levels)


taxonomy["Niveau_effectif"] = taxonomy.apply(effective_level, axis=1)


def effective_taxon_id(row: pd.Series) -> str:
    """Build a stable identifier from the complete path to the effective rank."""
    level = int(row["Niveau_effectif"])
    path = [str(row[rank]) for rank in TAXONOMY_RANKS[: level + 1]]
    return f"{TAXONOMY_RANKS[level]}::" + " > ".join(path)


taxonomy["Taxon_effectif_id"] = taxonomy.apply(effective_taxon_id, axis=1)
taxonomy["Taxon_effectif"] = taxonomy.apply(lambda row: row[TAXONOMY_RANKS[int(row["Niveau_effectif"])]], axis=1)
taxonomy["Classe_couverte"] = taxonomy[SOURCE_CLASS_COLUMN].notna()
# Toutes les lignes sont régionales ; x indique uniquement la fréquence.
regional_taxonomy = taxonomy.copy()

regional_taxa = (
    regional_taxonomy.sort_values(["Niveau_effectif", "Taxon_effectif_id"])
    .groupby("Taxon_effectif_id", sort=False, as_index=False)
    .agg(
        Niveau_effectif=("Niveau_effectif", "first"), Taxon_effectif=("Taxon_effectif", "first"),
        Nombre_images_originales=(IMAGE_COUNT_COLUMN, "sum"),
        Nombre_classes_originales=(SOURCE_CLASS_COLUMN, lambda values: values.dropna().nunique()),
        **{rank: (rank, "first") for rank in TAXONOMY_RANKS},
    )
)

quality_summary = pd.DataFrame({
    "Indicateur": ["Taxons régionaux distincts", "Lignes régionales de référence", "Classes originales couvrant la région", "Images originales pour la région"],
    "Valeur": [len(regional_taxa), len(regional_taxonomy), regional_taxonomy[SOURCE_CLASS_COLUMN].nunique(), int(regional_taxonomy[IMAGE_COUNT_COLUMN].sum())],
})
display(quality_summary)

## 3. Charger AquaMonitor complet

On commence par une ligne par individu, puis on applique les règles taxonomiques.

In [ ]:
# Le Parquet complet contient une ligne par image, mais pas les rangs
# taxonomiques. La table officielle les relie au libellé morphologique taxon_label.
AM_COLUMNS = ["img", "individual", "taxon", "taxon_group", "taxon_label"]
am_images = pd.read_parquet(am_path, columns=AM_COLUMNS)
taxon_table = pd.read_csv(taxon_table_path)
required_ranks = [*AM_RANKS, "species"]
if taxon_table["label"].duplicated().any():
    raise ValueError("La table officielle comporte des labels taxonomiques ambigus.")

# Les mentions de stade de vie sont des groupes visuels, pas des rangs de
# taxonomie. On les retire uniquement des colonnes structurées, sans modifier
# taxon_group, ni supposer de synonymie entre noms scientifiques.
for rank in required_ranks:
    taxon_table[rank] = taxon_table[rank].astype("string").str.replace(
        r"\s+(?:juv\.?|larv\.?|adult)$", "", regex=True
    ).str.strip()
am_images = am_images.merge(
    taxon_table[["label", *required_ranks]],
    left_on="taxon_label", right_on="label", how="left",
    validate="many_to_one", indicator="Taxonomie_source",
)
missing_taxonomy = am_images.loc[
    am_images["Taxonomie_source"].ne("both"), ["taxon_label", "taxon_group"]
].drop_duplicates()
if not missing_taxonomy.empty:
    display(missing_taxonomy)
    raise ValueError("Certains libellés n'ont pas de taxonomie officielle : vérifier la jointure.")
am_images = am_images.drop(columns=["label", "Taxonomie_source"])
if am_images["img"].duplicated().any() or am_images["individual"].isna().any():
    raise ValueError("Les images doivent être uniques et chaque image doit avoir un individu.")
print(f"Source complète : {len(am_images):,} images / "
      f"{am_images['individual'].nunique():,} individus / "
      f"{am_images['taxon_group'].nunique():,} classes")

individual_consistency = am_images.groupby("individual").agg(
    Nombre_groupes=("taxon_group", "nunique"), Nombre_images_AM_brutes=("img", "size")
)
if individual_consistency["Nombre_groupes"].max() > 1:
    raise ValueError("Au moins un individu AM possède plusieurs taxon_group.")
am_individuals = (
    am_images.sort_values("individual").drop_duplicates("individual")
    .merge(individual_consistency[["Nombre_images_AM_brutes"]], on="individual", how="left")
)

### Fonctions de correspondance

Ces fonctions vérifient un chemin taxonomique et choisissent la correspondance valide la plus précise.

In [ ]:
def compatible_with_regional_taxon(am_row: pd.Series, regional_row: pd.Series) -> bool:
    """Validate one AM taxonomy against one authoritative regional branch.

    Families must already exist. A determined genus must match exactly; a new
    genus is allowed only behind an indeterminate regional genus. More detailed
    species labels do not alter the structured REGNE-to-GENRE branch.
    """
    level = int(regional_row["Niveau_effectif"])
    if normalize_label(am_row[AM_RANKS[level]]) != normalize_label(regional_row[TAXONOMY_RANKS[level]]):
        return False
    for ancestor_level in range(level):
        expected = regional_row[TAXONOMY_RANKS[ancestor_level]]
        observed = am_row[AM_RANKS[ancestor_level]]
        if is_real_taxon(expected) and pd.notna(observed) and normalize_label(expected) != normalize_label(observed):
            return False
    # Ne jamais créer un nouveau rang frère si la référence contient déjà
    # une famille ou un genre déterminé au-delà de Niveau_precision.
    for descendant_level in range(level + 1, len(TAXONOMY_RANKS)):
        expected = regional_row[TAXONOMY_RANKS[descendant_level]]
        if is_real_taxon(expected):
            observed = am_row[AM_RANKS[descendant_level]]
            if pd.isna(observed) or normalize_label(expected) != normalize_label(observed):
                return False

    # Règle métier absolue : AquaMonitor ne peut jamais introduire une famille.
    # La famille doit déjà être déterminée dans la référence et être identique.
    reference_family = regional_row["FAMILLE"]
    observed_family = am_row["family"]
    if pd.notna(observed_family):
        if not is_real_taxon(reference_family):
            return False
        if normalize_label(reference_family) != normalize_label(observed_family):
            return False
    return True


def match_individual(row: pd.Series) -> pd.Series:
    """Assign one deduplicated AM individual to its most precise valid regional taxon.

    Returns an explicit ``EXCLU`` status when no branch satisfies every rule;
    exclusions are retained for audit instead of being silently discarded.
    """
    # Le test se fait sur les lignes originales, car un même taxon effectif
    # peut autoriser plusieurs descendants explicitement renseignés.
    mask = regional_taxonomy.apply(lambda ref: compatible_with_regional_taxon(row, ref), axis=1)
    candidates = regional_taxonomy[mask].sort_values("Niveau_effectif", ascending=False)
    if candidates.empty:
        return pd.Series({"Taxon_effectif_id": pd.NA, "Taxon_effectif": pd.NA, "Rang_correspondance": pd.NA, "Type_correspondance": "EXCLU"})
    match = candidates.iloc[0]
    level = int(match["Niveau_effectif"])
    match_type = "EXACT" if normalize_label(row["taxon_group"]) == normalize_label(match["Taxon_effectif"]) else "DETAIL_AJOUTE"
    return pd.Series({
        "Taxon_effectif_id": match["Taxon_effectif_id"], "Taxon_effectif": match["Taxon_effectif"],
        "Rang_correspondance": TAXONOMY_RANKS[level], "Type_correspondance": match_type,
    })

### Appliquer le filtre

Les individus refusés sont conservés dans un tableau de contrôle.

In [ ]:
# Une signature regroupe tous les individus ayant la même taxonomie.
# Le résultat du filtre est ensuite propagé par une jointure contrôlée.
signature_columns = ["taxon_group", *AM_RANKS, "species"]
signatures = am_individuals[signature_columns].drop_duplicates().reset_index(drop=True)
signature_matches = signatures.apply(match_individual, axis=1)
signatures = pd.concat([signatures, signature_matches], axis=1)
am_individuals = am_individuals.merge(
    signatures, on=signature_columns, how="left", validate="many_to_one"
)
assert len(am_individuals) == am_images["individual"].nunique()
am_individuals["Retenu_region"] = am_individuals["Type_correspondance"].ne("EXCLU")
matched_individuals = am_individuals[am_individuals["Retenu_region"]].copy()
excluded_individuals = am_individuals[~am_individuals["Retenu_region"]].copy()
filtered_am_images = am_images[am_images["individual"].isin(set(matched_individuals["individual"]))].copy()

filter_summary = pd.DataFrame({
    "Indicateur": ["Images AM brutes", "Individus AM", "Individus retenus", "Individus exclus", "Classes AM retenues", "Images brutes des individus retenus"],
    "Valeur": [len(am_images), am_images["individual"].nunique(), matched_individuals["individual"].nunique(), excluded_individuals["individual"].nunique(), matched_individuals["taxon_group"].nunique(), len(filtered_am_images)],
})
excluded_by_class = (
    excluded_individuals.groupby("taxon_group", dropna=False)
    .agg(Nombre_individus=("individual", "nunique"), Nombre_images_AM_brutes=("Nombre_images_AM_brutes", "sum"))
    .reset_index().sort_values("Nombre_individus", ascending=False)
)
display(filter_summary)
display(excluded_by_class)

## 4. Apport par taxon

On affiche seulement les taxons qui reçoivent au moins un individu.

In [ ]:
added_by_taxon = (
    matched_individuals.groupby("Taxon_effectif_id")
    .agg(Individus_AquaMonitor=("individual", "nunique"), Classes_AquaMonitor=("taxon_group", "nunique"), Images_AM_brutes_diagnostic=("Nombre_images_AM_brutes", "sum"))
    .reset_index()
)
taxon_analysis = regional_taxa.merge(added_by_taxon, on="Taxon_effectif_id", how="left")
for column in ["Individus_AquaMonitor", "Classes_AquaMonitor", "Images_AM_brutes_diagnostic"]:
    taxon_analysis[column] = taxon_analysis[column].fillna(0).astype("int64")
taxon_analysis["Total_comparable"] = taxon_analysis["Nombre_images_originales"] + taxon_analysis["Individus_AquaMonitor"]

taxon_plot = taxon_analysis[taxon_analysis["Individus_AquaMonitor"] > 0].copy()
taxon_plot["Libelle"] = taxon_plot["Taxon_effectif"] + " (" + taxon_plot.apply(lambda row: TAXONOMY_RANKS[int(row["Niveau_effectif"])].title(), axis=1) + ")"
taxon_plot = taxon_plot.sort_values("Total_comparable")
fig, ax = plt.subplots(figsize=(12, max(6, 0.55 * len(taxon_plot))))
ax.barh(taxon_plot["Libelle"], taxon_plot["Total_comparable"], color="#4472C4", label="Original + individus AquaMonitor")
ax.barh(taxon_plot["Libelle"], taxon_plot["Nombre_images_originales"], color="#ED7D31", label="Images originales")
ax.set_xlabel("Volume comparable (1 individu AquaMonitor = 1 ajout)")
ax.set_title("Taxons régionaux enrichis par AquaMonitor complet")
ax.legend(); ax.grid(axis="x", alpha=0.2); fig.tight_layout()
fig.savefig(reports_dir / "am_taxons_original_vs_enrichi.png", dpi=180, bbox_inches="tight")
plt.show()
display(taxon_plot.sort_values("Individus_AquaMonitor", ascending=False))

## 5. Apport par classe

Une classe est soit retrouvée dans le fichier original, soit ajoutée sous un taxon déjà accepté.

In [ ]:
def normalized_am_class(value) -> str | None:
    """Normalize a AM class and ignore the non-taxonomic ``adult`` suffix."""
    key = normalize_label(value)
    return key[:-6] if key and key.endswith("_adult") else key


def class_suffix_matches(source_class, am_class) -> bool:
    """Check whether a AM label maps to an existing reference class convention."""
    source_key, am_key = normalize_label(source_class), normalized_am_class(am_class)
    if not source_key or not am_key:
        return False
    return (
        source_key.endswith("_" + am_key)
        or source_key.endswith("_" + am_key + "_sp")
        # Une classe AM déterminée à la famille correspond à la classe
        # générique « Famille_Genus_sp » déjà présente dans la référence.
        or source_key.endswith("_" + am_key + "_genus_sp")
    )

In [ ]:
base_classes = regional_taxonomy[regional_taxonomy[SOURCE_CLASS_COLUMN].notna()].copy()
am_class_rows = (
    matched_individuals.groupby(["Taxon_effectif_id", "taxon_group"], dropna=False)
    .agg(Individus_AquaMonitor=("individual", "nunique")).reset_index()
)
class_assignments = []
for _, am_class in am_class_rows.iterrows():
    candidates = base_classes[
        base_classes["Taxon_effectif_id"].eq(am_class["Taxon_effectif_id"])
        & base_classes[SOURCE_CLASS_COLUMN].map(lambda value: class_suffix_matches(value, am_class["taxon_group"]))
    ]
    source_class = candidates.iloc[0][SOURCE_CLASS_COLUMN] if len(candidates) == 1 else pd.NA
    class_assignments.append({
        "Taxon_effectif_id": am_class["Taxon_effectif_id"], "Classe_AquaMonitor": am_class["taxon_group"],
        "Classe_originale_associee": source_class, "Individus_AquaMonitor": int(am_class["Individus_AquaMonitor"]),
    })
class_assignments = pd.DataFrame(class_assignments)

base_class_analysis = base_classes[["Taxon_effectif_id", SOURCE_CLASS_COLUMN, IMAGE_COUNT_COLUMN, *TAXONOMY_RANKS]].rename(columns={IMAGE_COUNT_COLUMN: "Images_originales"})
found = class_assignments[class_assignments["Classe_originale_associee"].notna()].groupby("Classe_originale_associee", as_index=False).agg(
    Individus_AquaMonitor=("Individus_AquaMonitor", "sum"),
    Classes_AquaMonitor_retrouvees=("Classe_AquaMonitor", lambda values: " | ".join(sorted(values.unique()))),
)
base_class_analysis = base_class_analysis.merge(found, how="left", left_on=SOURCE_CLASS_COLUMN, right_on="Classe_originale_associee")
base_class_analysis["Individus_AquaMonitor"] = base_class_analysis["Individus_AquaMonitor"].fillna(0).astype("int64")
base_class_analysis["Provenance"] = np.where(base_class_analysis["Individus_AquaMonitor"] > 0, "Base + AquaMonitor", "Base")
base_class_analysis["Classe_affichee"] = base_class_analysis[SOURCE_CLASS_COLUMN]

new_classes = class_assignments[class_assignments["Classe_originale_associee"].isna()].merge(regional_taxa[["Taxon_effectif_id", *TAXONOMY_RANKS]], on="Taxon_effectif_id", how="left")
new_classes["Images_originales"] = 0
new_classes["Provenance"] = "AquaMonitor"
new_classes["Classe_affichee"] = "AquaMonitor · " + new_classes["Classe_AquaMonitor"].astype(str)

analysis_columns = ["Taxon_effectif_id", "Classe_affichee", "Images_originales", "Individus_AquaMonitor", "Provenance", *TAXONOMY_RANKS]
class_analysis = pd.concat([base_class_analysis[analysis_columns], new_classes[analysis_columns]], ignore_index=True)
class_analysis["Total_comparable"] = class_analysis["Images_originales"] + class_analysis["Individus_AquaMonitor"]
class_plot = class_analysis[class_analysis["Individus_AquaMonitor"] > 0].sort_values("Total_comparable")
fig, ax = plt.subplots(figsize=(13, max(6, 0.5 * len(class_plot))))
ax.barh(class_plot["Classe_affichee"], class_plot["Total_comparable"], color="#4472C4", label="Original + individus AquaMonitor")
ax.barh(class_plot["Classe_affichee"], class_plot["Images_originales"], color="#ED7D31", label="Images originales")
ax.set_xlabel("Volume comparable (1 individu AquaMonitor = 1 ajout)")
ax.set_title("Classes retrouvées ou ajoutées par AquaMonitor complet")
ax.legend(); ax.grid(axis="x", alpha=0.2); fig.tight_layout()
fig.savefig(reports_dir / "am_classes_original_vs_enrichi.png", dpi=180, bbox_inches="tight")
plt.show()
display(class_plot.sort_values("Individus_AquaMonitor", ascending=False))

## 6. Regarder quelques individus (optionnel)

Choisir une classe et une page dans la cellule suivante. Une seule image est
affichée par individu. Le chargement des pixels peut demander une connexion réseau.

In [ ]:
# Configuration du contrôle visuel. Le tableau affiché plus bas donne les valeurs
# autorisées pour CLASSE_A_VISUALISER.
DATASET_AM_ID = "mikkoim/aquamonitor"
CLASSE_A_VISUALISER = None  # Exemple : "Leptophlebia"
PAGE_A_VISUALISER = 2
INDIVIDUS_PAR_PAGE = 20
CHARGER_APERCU = False

In [ ]:
def build_visual_class_catalog(assignments: pd.DataFrame) -> pd.DataFrame:
    """Build the auditable list of AquaMonitor classes eligible for inspection.

    A class is ``Enrichie`` when it maps to an existing class from the original
    workbook. It is ``Ajoutée`` when the validated AquaMonitor detail created a
    new dataset-class row without changing the authoritative REGNE-to-GENRE path.
    """
    catalog = assignments.copy()
    catalog["Statut_classe"] = np.where(
        catalog["Classe_originale_associee"].notna(), "Enrichie", "Ajoutée"
    )
    return catalog[[
        "Classe_AquaMonitor", "Statut_classe", "Classe_originale_associee",
        "Individus_AquaMonitor",
    ]].sort_values(["Statut_classe", "Classe_AquaMonitor"]).reset_index(drop=True)


def representative_images_for_class(
    class_name: str,
    page: int = 1,
    individuals_per_page: int = 20,
) -> tuple[pd.DataFrame, int]:
    """Select one deterministic image filename per retained individual.

    Selection uses the first filename after sorting by ``individual`` and ``img``.
    It operates only on ``filtered_am_images``, so excluded taxa cannot enter the
    visual audit. The returned DataFrame covers one requested display page.
    """
    if page < 1:
        raise ValueError("PAGE_A_VISUALISER doit être supérieur ou égal à 1.")
    if individuals_per_page < 1:
        raise ValueError("INDIVIDUS_PAR_PAGE doit être supérieur ou égal à 1.")

    class_images = filtered_am_images.loc[
        filtered_am_images["taxon_group"].eq(class_name),
        ["individual", "taxon_group", "img"],
    ].sort_values(["individual", "img"])
    representatives = class_images.drop_duplicates("individual", keep="first")
    total_pages = max(1, int(np.ceil(len(representatives) / individuals_per_page)))
    if page > total_pages:
        raise ValueError(
            f"Page {page} inexistante pour {class_name!r} : "
            f"choisir une page entre 1 et {total_pages}."
        )
    start = (page - 1) * individuals_per_page
    return representatives.iloc[start : start + individuals_per_page].copy(), total_pages


def stream_selected_images(target_rows: pd.DataFrame) -> dict[str, object]:
    """Read selected image pixels from the Hugging Face WebDataset stream.

    Matching is performed with the filename stem because the stream ``__key__``
    can omit the ``.jpg`` extension. Iteration stops as soon as every requested
    representative has been found.
    """
    from datasets import load_dataset

    target_by_stem = {
        Path(filename).stem: filename for filename in target_rows["img"].astype(str)
    }
    images_by_filename: dict[str, object] = {}
    # Les archives thumbnail sont beaucoup plus petites que les originaux.
    # La lecture reste optionnelle : les noms sont distribués entre 111 archives.
    from huggingface_hub import HfApi
    archives = [
        name for name in HfApi().list_repo_files(
            DATASET_AM_ID, repo_type="dataset", revision=DATASET_REVISION
        ) if name.startswith("thumbnail/") and name.endswith(".tar")
    ]
    image_stream = load_dataset(
        "webdataset",
        data_files={"train": [
            f"https://huggingface.co/datasets/{DATASET_AM_ID}/resolve/"
            f"{DATASET_REVISION}/{name}" for name in sorted(archives)
        ]},
        split="train", streaming=True,
        cache_dir=str(experiment_dir / "data" / "hf_cache"),
    )
    for sample in image_stream:
        target_filename = target_by_stem.get(Path(str(sample.get("__key__", ""))).stem)
        if target_filename is None:
            continue
        image_value = next(
            (value for value in sample.values()
             if hasattr(value, "size") and hasattr(value, "convert")), None,
        )
        if image_value is not None:
            images_by_filename[target_filename] = image_value
        if len(images_by_filename) == len(target_by_stem):
            break
    return images_by_filename



def display_individual_gallery(
    target_rows: pd.DataFrame,
    images_by_filename: dict[str, object],
    class_name: str,
    class_status: str,
    page: int,
    total_pages: int,
) -> None:
    """Render a compact gallery with one representative image per individual."""
    if target_rows.empty:
        print(f"Aucun individu retenu pour la classe {class_name!r}.")
        return

    columns = min(4, len(target_rows))
    rows = int(np.ceil(len(target_rows) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(4 * columns, 3.4 * rows), squeeze=False)
    for axis, (_, record) in zip(axes.flat, target_rows.iterrows()):
        image = images_by_filename.get(str(record["img"]))
        if image is None:
            axis.text(0.5, 0.5, "Image non retrouvée dans le flux", ha="center", va="center")
        else:
            axis.imshow(image)
        axis.set_title(f"Individu : {record['individual']}", fontsize=9)
        axis.axis("off")
    for axis in axes.flat[len(target_rows):]:
        axis.axis("off")
    fig.suptitle(
        f"{class_name} — classe {class_status.lower()} — page {page}/{total_pages}",
        fontsize=13,
    )
    fig.tight_layout()
    plt.show()

In [ ]:
visual_class_catalog = build_visual_class_catalog(class_assignments)
display(visual_class_catalog)

available_visual_classes = visual_class_catalog["Classe_AquaMonitor"].tolist()
if not available_visual_classes:
    print("Aucune classe AquaMonitor ajoutée ou enrichie à visualiser.")
elif not CHARGER_APERCU:
    print("Aperçu désactivé : mettre CHARGER_APERCU = False pour charger les images.")
else:
    selected_class = CLASSE_A_VISUALISER or available_visual_classes[0]
    if selected_class not in available_visual_classes:
        raise ValueError(
            f"Classe inconnue : {selected_class!r}. Utiliser une valeur du tableau ci-dessus."
        )
    selected_status = visual_class_catalog.loc[
        visual_class_catalog["Classe_AquaMonitor"].eq(selected_class), "Statut_classe"
    ].iloc[0]
    representative_page, number_of_pages = representative_images_for_class(
        selected_class,
        page=PAGE_A_VISUALISER,
        individuals_per_page=INDIVIDUS_PAR_PAGE,
    )
    selected_images = stream_selected_images(representative_page)
    print(
        f"Classe : {selected_class} ({selected_status}) | "
        f"page {PAGE_A_VISUALISER}/{number_of_pages} | "
        f"{len(selected_images)}/{len(representative_page)} images trouvées"
    )
    display_individual_gallery(
        representative_page,
        selected_images,
        selected_class,
        selected_status,
        PAGE_A_VISUALISER,
        number_of_pages,
    )

## 7. Graphiques simples

Ces graphiques montrent le nombre d'individus retenus par taxon et par classe.

In [ ]:
individuals_by_taxon = matched_individuals.groupby(["Taxon_effectif_id", "Taxon_effectif"]).agg(Nombre_individus=("individual", "nunique")).reset_index().sort_values("Nombre_individus")
fig, ax = plt.subplots(figsize=(11, max(5, 0.5 * len(individuals_by_taxon))))
ax.barh(individuals_by_taxon["Taxon_effectif"], individuals_by_taxon["Nombre_individus"], color="#4472C4")
ax.set_title("Individus AquaMonitor retenus par taxon régional"); ax.set_xlabel("Nombre d'individus"); ax.grid(axis="x", alpha=0.2); fig.tight_layout()
fig.savefig(reports_dir / "am_individus_par_taxon.png", dpi=180, bbox_inches="tight"); plt.show()

individuals_by_class = matched_individuals.groupby("taxon_group").agg(Nombre_individus=("individual", "nunique")).reset_index().sort_values("Nombre_individus")
fig, ax = plt.subplots(figsize=(11, max(6, 0.45 * len(individuals_by_class))))
ax.barh(individuals_by_class["taxon_group"], individuals_by_class["Nombre_individus"], color="#70AD47")
ax.set_title("Individus AquaMonitor retenus par classe AM"); ax.set_xlabel("Nombre d'individus"); ax.grid(axis="x", alpha=0.2); fig.tight_layout()
fig.savefig(reports_dir / "am_individus_par_classe.png", dpi=180, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(matched_individuals["Nombre_images_AM_brutes"], bins=30, color="#A5A5A5", edgecolor="white")
ax.set_title("Images brutes par individu AquaMonitor retenu (diagnostic)"); ax.set_xlabel("Nombre d'images pour un individu"); ax.set_ylabel("Nombre d'individus"); fig.tight_layout()
fig.savefig(reports_dir / "am_images_par_individu.png", dpi=180, bbox_inches="tight"); plt.show()

## 8. Préparer les fonctions d'export

On construit l'arbre et les fonctions de mise en forme Excel.

In [ ]:
tree_base = base_class_analysis[[*TAXONOMY_RANKS, SOURCE_CLASS_COLUMN, "Images_originales", "Individus_AquaMonitor", "Provenance", "Taxon_effectif_id"]].rename(columns={SOURCE_CLASS_COLUMN: "CLASSE DATASET"})
tree_without_class = regional_taxonomy[regional_taxonomy[SOURCE_CLASS_COLUMN].isna()][[*TAXONOMY_RANKS, "Taxon_effectif_id"]].drop_duplicates("Taxon_effectif_id")
tree_without_class["CLASSE DATASET"] = "— aucune classe originale —"
tree_without_class["Images_originales"] = 0
tree_without_class["Individus_AquaMonitor"] = 0
tree_without_class["Provenance"] = "Référence régionale"
tree_new = new_classes[[*TAXONOMY_RANKS, "Classe_affichee", "Images_originales", "Individus_AquaMonitor", "Provenance", "Taxon_effectif_id"]].rename(columns={"Classe_affichee": "CLASSE DATASET"})
horizontal_tree = pd.concat([tree_base, tree_without_class, tree_new], ignore_index=True)
horizontal_tree["TOTAL COMPARABLE"] = horizontal_tree["Images_originales"] + horizontal_tree["Individus_AquaMonitor"]
horizontal_tree["FRÉQUENT BASSIN AG"] = "x"
horizontal_tree = horizontal_tree.sort_values([*TAXONOMY_RANKS, "CLASSE DATASET"]).reset_index(drop=True)

# Même invariant dans l'arbre régional : les nouvelles classes peuvent
# enrichir les feuilles, jamais créer une branche REGNE→GENRE.
regional_branch_set = set(
    regional_taxonomy[TAXONOMY_RANKS].fillna("<VIDE>").astype(str).itertuples(index=False, name=None)
)
tree_branch_set = set(
    horizontal_tree[TAXONOMY_RANKS].fillna("<VIDE>").astype(str).itertuples(index=False, name=None)
)
assert tree_branch_set == regional_branch_set

In [ ]:
def aggregate_level(frame: pd.DataFrame, level_index: int) -> pd.DataFrame:
    """Aggregate class and volume metrics at one rank of the taxonomy tree."""
    path = TAXONOMY_RANKS[:level_index + 1]
    return frame.groupby(path, dropna=False).agg(
        Nombre_taxons_regionaux=("Taxon_effectif_id", "nunique"), Nombre_classes=("CLASSE DATASET", "nunique"),
        Images_originales=("Images_originales", "sum"), Individus_AquaMonitor=("Individus_AquaMonitor", "sum"),
        Total_comparable=("TOTAL COMPARABLE", "sum"),
    ).reset_index()


level_tables = {rank: aggregate_level(horizontal_tree, i) for i, rank in enumerate(TAXONOMY_RANKS)}

In [ ]:
def style_workbook(path: Path) -> None:
    """Apply deterministic presentation rules to the hierarchical workbook.

    The function changes only formatting and merged display cells; analytical
    values are computed before this step.
    """
    from openpyxl import load_workbook
    from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
    from openpyxl.utils import get_column_letter

    workbook = load_workbook(path)
    header_fill, header_font = PatternFill("solid", fgColor="1F4E78"), Font(color="FFFFFF", bold=True)
    for worksheet in workbook.worksheets:
        worksheet.freeze_panes = "A2"; worksheet.sheet_view.showGridLines = False; worksheet.auto_filter.ref = worksheet.dimensions
        for cell in worksheet[1]:
            cell.fill = header_fill; cell.font = header_font; cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        for cells in worksheet.columns:
            values = [str(cell.value or "") for cell in cells[:200]]
            worksheet.column_dimensions[get_column_letter(cells[0].column)].width = min(max(max(map(len, values), default=0) + 2, 11), 44)

    worksheet = workbook["Arbre_horizontal"]
    worksheet.auto_filter.ref = None; worksheet.sheet_view.zoomScale = 70; worksheet.page_setup.orientation = "landscape"
    worksheet.sheet_properties.pageSetUpPr.fitToPage = True; worksheet.page_setup.fitToWidth = 1; worksheet.page_setup.fitToHeight = 0
    palette = ["D9EAF7", "DDEBF7", "E2F0D9", "FFF2CC", "FCE4D6", "E4DFEC"]
    thin_black = Side(style="thin", color="000000"); border = Border(left=thin_black, right=thin_black, top=thin_black, bottom=thin_black)
    for level_index, rank in enumerate(TAXONOMY_RANKS):
        keys = [tuple(str(horizontal_tree.iloc[row][r]) for r in TAXONOMY_RANKS[:level_index + 1]) for row in range(len(horizontal_tree))]
        start = 0
        while start < len(keys):
            end = start
            while end + 1 < len(keys) and keys[end + 1] == keys[start]: end += 1
            group = horizontal_tree.iloc[start:end + 1]
            top = worksheet.cell(start + 2, level_index + 1)
            top.value = (f"{keys[start][-1]}\n{int(group['Images_originales'].sum()):,} orig. · {int(group['Individus_AquaMonitor'].sum()):,} ind. AM · {group['Taxon_effectif_id'].nunique()} taxon(s)").replace(",", " ")
            if end > start: worksheet.merge_cells(start_row=start + 2, start_column=level_index + 1, end_row=end + 2, end_column=level_index + 1)
            top.fill = PatternFill("solid", fgColor=palette[level_index]); top.font = Font(bold=level_index <= 1, size=9)
            top.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True); top.border = border
            start = end + 1
    headers = {cell.value: cell.column for cell in worksheet[1]}
    provenance_colors = {
        "Base": "F2F2F2", "Base + AquaMonitor": "C6E0B4",
        "Original": "F2F2F2", "Original + AquaMonitor": "C6E0B4",
        "AquaMonitor": "BDD7EE", "Référence régionale": "FFF2CC",
    }
    individual_header = "NB INDIVIDUS" if "NB INDIVIDUS" in headers else "TOTAL COMPARABLE"
    frequency_header = "FRÉQUENT BASSIN AG"
    zero_fill = PatternFill("solid", fgColor="F4CCCC")
    for row in range(2, worksheet.max_row + 1):
        for name in ["Images_originales", "Individus_AquaMonitor", individual_header]:
            worksheet.cell(row, headers[name]).number_format = "#,##0"
        source = worksheet.cell(row, headers["Provenance"]); source.fill = PatternFill("solid", fgColor=provenance_colors.get(source.value, "FFFFFF"))
        for col in range(7, worksheet.max_column + 1): worksheet.cell(row, col).alignment = Alignment(vertical="center", wrap_text=True)
        worksheet.cell(row, headers[individual_header]).alignment = Alignment(horizontal="center", vertical="center")
        worksheet.cell(row, headers[frequency_header]).alignment = Alignment(horizontal="center", vertical="center")
        if (worksheet.cell(row, headers[individual_header]).value or 0) == 0:
            worksheet.cell(row, headers[individual_header]).fill = zero_fill
        worksheet.row_dimensions[row].height = 34
    for index, width in enumerate([20, 22, 20, 23, 25, 23, 52, 15, 18, 17, 22, 17], start=1): worksheet.column_dimensions[get_column_letter(index)].width = width
    worksheet.row_dimensions[1].height = 34
    # À l'ouverture, Excel affiche directement l'arbre demandé par l'utilisateur.
    workbook.active = workbook.sheetnames.index("Arbre_horizontal")
    workbook.save(path)


def export_filterable_tree(reference: pd.DataFrame, destination: Path) -> None:
    """Exporter deux vues du même arbre, sans feuilles d'analyse annexes.

    La feuille Arbre fusionne les catégories pour une lecture hiérarchique.
    Vue_filtrable garde les lignes autonomes pour le tri et les filtres Excel.
    La provenance reste explicite ; les effectifs sont calculés en amont.
    """
    from openpyxl import Workbook
    from copy import copy
    from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
    from openpyxl.formatting.rule import CellIsRule
    from openpyxl.utils import get_column_letter

    rank_columns = ["REGNE", "EMBRANCHEMENT", "CLASSE", "ORDRE", "FAMILLE", "GENRE"]
    selected_columns = [
        *rank_columns, "Classe", "Provenance", "Nb individus total",
        "Fréquent en Bassin AG",
    ]
    tree = reference[selected_columns].sort_values(
        [*rank_columns, "Classe"], na_position="last"
    ).copy()
    tree["Classe"] = tree["Classe"].fillna("— aucune classe dataset —")
    tree["Fréquent en Bassin AG"] = tree["Fréquent en Bassin AG"].fillna("")
    workbook = Workbook()
    sheet = workbook.active
    sheet.title = "Vue_filtrable"
    sheet.append([
        *rank_columns, "CLASSE DATASET", "PROVENANCE", "NB INDIVIDUS",
        "FRÉQUENT BASSIN AG",
    ])
    for record in tree.itertuples(index=False, name=None):
        sheet.append([None if pd.isna(value) else value for value in record])

    black = Side(style="thin", color="000000")
    borders = Border(left=black, right=black, top=black, bottom=black)
    palette = ["D9EAF7", "DDEBF7", "E2F0D9", "FFF2CC", "FCE4D6", "E4DFEC"]
    provenance_colors = {
        "Original": "FCE4D6", "AquaMonitor": "D9EAF7",
        "Original + AquaMonitor": "E2F0D9",
    }
    for row in sheet:
        for cell in row:
            cell.border = borders
            cell.alignment = Alignment(vertical="center", wrap_text=True)
    for cell in sheet[1]:
        cell.fill = PatternFill("solid", fgColor="1F4E78")
        cell.font = Font(bold=True, color="FFFFFF")
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for row_number in range(2, sheet.max_row + 1):
        for column_number, color in enumerate(palette, start=1):
            sheet.cell(row_number, column_number).fill = PatternFill("solid", fgColor=color)
        source = sheet.cell(row_number, 8).value
        sheet.cell(row_number, 7).fill = PatternFill("solid", fgColor=provenance_colors[source])
        sheet.cell(row_number, 9).number_format = "#,##0"
        if sheet.cell(row_number, 9).value == 0:
            sheet.cell(row_number, 9).fill = PatternFill("solid", fgColor="F4CCCC")
        for column_number in (9, 10):
            sheet.cell(row_number, column_number).alignment = Alignment(
                horizontal="center", vertical="center", wrap_text=True
            )
        sheet.row_dimensions[row_number].height = 32
    if sheet.max_row >= 2:
        sheet.conditional_formatting.add(
            f"I2:I{sheet.max_row}",
            CellIsRule(operator="equal", formula=["0"],
                       fill=PatternFill("solid", fgColor="F4CCCC")),
        )
    widths = [14, 24, 22, 24, 25, 24, 72, 25, 20, 21]
    for index, width in enumerate(widths, start=1):
        sheet.column_dimensions[get_column_letter(index)].width = width
    sheet.row_dimensions[1].height = 36
    sheet.freeze_panes = "G2"
    sheet.auto_filter.ref = sheet.dimensions
    sheet.sheet_view.showGridLines = False
    sheet.sheet_view.zoomScale = 75
    # La copie garde chaque classe et ses effectifs ; seules les six colonnes
    # taxonomiques sont fusionnées par groupes contigus de même chemin ancestral.
    visual = workbook.copy_worksheet(sheet)
    visual.title = "Arbre"
    visual.auto_filter.ref = None
    visual.freeze_panes = "A2"
    for level, rank in enumerate(rank_columns, start=1):
        paths = list(tree[rank_columns[:level]].fillna("— non renseigné —").astype(str).itertuples(index=False, name=None))
        start = 0
        while start < len(paths):
            end = start
            while end + 1 < len(paths) and paths[end + 1] == paths[start]:
                end += 1
            total = int(tree.iloc[start:end + 1]["Nb individus total"].fillna(0).sum())
            first_row, last_row = start + 2, end + 2
            if last_row > first_row:
                visual.merge_cells(start_row=first_row, end_row=last_row,
                                   start_column=level, end_column=level)
            cell = visual.cell(first_row, level)
            cell.value = f"{paths[start][-1]}\n{total:,} individus".replace(",", " ")
            cell.alignment = Alignment(horizontal="center", vertical="top" if end - start > 12 else "center", wrap_text=True)
            cell.font = Font(bold=level <= 2, size=10)
            cell.fill = PatternFill("solid", fgColor="F4CCCC" if total == 0 else palette[level - 1])
            cell.border = borders
            # Les bordures noires entourent le groupe, sans lignes horizontales
            # à l'intérieur de la grande case fusionnée.
            for row in range(first_row + 1, last_row + 1):
                edge = visual.cell(row, level)
                edge.border = Border(left=black, right=black,
                                     bottom=black if row == last_row else Side())
                edge.fill = copy(cell.fill)
            start = end + 1
    visual.conditional_formatting.add(
        f"I2:I{visual.max_row}",
        CellIsRule(operator="equal", formula=["0"],
                   fill=PatternFill("solid", fgColor="F4CCCC")),
    )
    workbook.move_sheet(visual, offset=-1)
    workbook.active = 0
    destination.parent.mkdir(parents=True, exist_ok=True)
    workbook.save(destination)

In [ ]:
output_path = reports_dir / "taxonomie_hierarchique_aquamonitor_complet_arbre_seul.xlsx"

## 9. Construire le référentiel enrichi

On repart de toutes les lignes originales et on ajoute uniquement les classes valides.

In [ ]:
# La table finale repart de toutes les lignes originales, sans les filtrer à la région.
BASE_COLUMNS = [
    "Classe", "REGNE", "EMBRANCHEMENT", "CLASSE", "ORDRE", "FAMILLE", "GENRE",
    "Commentaires", "Fréquent en Bassin AG", "Niveau_precision", "Nombre d'images",
]
complete_reference = pd.read_excel(
    reference_path, sheet_name="Sheet1", usecols=BASE_COLUMNS, engine="openpyxl"
)

# Une ligne statistique par taxon_group AM.
for rank in AM_RANKS:
    max_values = am_images.groupby("taxon_group")[rank].nunique(dropna=True).max()
    if max_values > 1:
        raise ValueError(f"Une classe AM possède plusieurs valeurs pour le rang {rank}.")

am_classes_export = (
    am_images.groupby("taxon_group", dropna=False)
    .agg(
        Nombre_individus=("individual", "nunique"),
        Nombre_images_brutes=("img", "size"),
        **{rank: (rank, "first") for rank in AM_RANKS},
    )
    .reset_index()
)

In [ ]:
def converted_class_name(row: pd.Series) -> str:
    """Build a class label following the reference workbook naming convention.

    The label may expose genus/species detail, while structured taxonomy columns
    remain copied from an existing regional combination.
    """
    # Reproduit la convention EMBRANCHEMENT_CLASSE_ORDRE_FAMILLE_GENRE_espece."""
    path = []
    for rank in ["phylum", "class", "order", "family", "genus"]:
        if pd.notna(row[rank]):
            path.extend(str(row[rank]).strip().replace(" ", "_").split("_"))

    group_tokens = str(row["taxon_group"]).strip().replace(" ", "_").split("_")
    if pd.notna(row["genus"]):
        genus_key = normalize_label(row["genus"])
        if normalize_label(group_tokens[0]) == genus_key:
            detail = group_tokens[1:] or ["sp"]
        else:
            detail = group_tokens
    elif pd.notna(row["family"]) and normalize_label(row["taxon_group"]) == normalize_label(row["family"]):
        detail = ["Genus", "sp"]
    elif path and normalize_label(row["taxon_group"]) == normalize_label(path[-1]):
        detail = ["sp"]
    else:
        detail = group_tokens
    return "_".join([*path, *detail])


def am_precision(row: pd.Series) -> int:
    """Return the deepest populated AM rank using the reference zero-based scale."""
    present = [i for i, rank in enumerate(AM_RANKS) if pd.notna(row[rank])]
    return max(present) if present else 0

### Calculer les apports par classe

In [ ]:
# Volumes AquaMonitor uniquement pour les classes ayant passé le filtre régional strict.
am_valid_stats = (
    matched_individuals.groupby("taxon_group", as_index=False)
    .agg(
        Nb_individus_AquaMonitor=("individual", "nunique"),
        Nb_images_AquaMonitor=("Nombre_images_AM_brutes", "sum"),
    )
    .merge(class_assignments, left_on="taxon_group", right_on="Classe_AquaMonitor", how="left", validate="one_to_one")
    .merge(am_classes_export, on="taxon_group", how="left", validate="one_to_one")
)

# Deuxième garde-fou : si le libellé converti existe exactement, il s'agit
# d'une classe commune et non d'une nouvelle ligne.
am_valid_stats["Classe_convertie"] = am_valid_stats.apply(converted_class_name, axis=1)
original_class_names = set(complete_reference["Classe"].dropna())
fallback_existing = (
    am_valid_stats["Classe_originale_associee"].isna()
    & am_valid_stats["Classe_convertie"].isin(original_class_names)
)
am_valid_stats.loc[fallback_existing, "Classe_originale_associee"] = am_valid_stats.loc[
    fallback_existing, "Classe_convertie"
]

### Enrichir les lignes existantes et ajouter les nouvelles classes

In [ ]:
# Hypothèse validée : une image originale correspond à un individu original distinct.
merged_reference = complete_reference.rename(columns={"Nombre d'images": "Nb images original"}).copy()
merged_reference["Provenance"] = "Original"
merged_reference["Nb individus original"] = pd.to_numeric(
    merged_reference["Nb images original"], errors="coerce"
).astype("Int64")
merged_reference["Nb individus AquaMonitor"] = 0
merged_reference["Nb images AquaMonitor"] = 0

# Enrichir les classes déjà présentes dans la référence.
existing_additions = am_valid_stats[am_valid_stats["Classe_originale_associee"].notna()].copy()
for _, addition in existing_additions.iterrows():
    mask = merged_reference["Classe"].eq(addition["Classe_originale_associee"])
    if mask.sum() != 1:
        raise ValueError(f"Classe originale introuvable ou ambiguë : {addition['Classe_originale_associee']}")
    merged_reference.loc[mask, "Provenance"] = "Original + AquaMonitor"
    merged_reference.loc[mask, "Nb individus AquaMonitor"] += int(addition["Nb_individus_AquaMonitor"])
    merged_reference.loc[mask, "Nb images AquaMonitor"] += int(addition["Nb_images_AquaMonitor"])

# Ajouter uniquement les classes AquaMonitor valides qui n'existent pas déjà.
new_rows = []
new_additions = am_valid_stats[am_valid_stats["Classe_originale_associee"].isna()].copy()
for _, addition in new_additions.iterrows():
    new_class_name = addition["Classe_convertie"]
    if merged_reference["Classe"].eq(new_class_name).any():
        raise ValueError(f"La nouvelle classe existe déjà : {new_class_name}")

    # La classe peut être plus détaillée dans son nom, mais elle doit être
    # rattachée à une combinaison REGNE→GENRE strictement existante.
    branch_candidates = regional_taxonomy[
        regional_taxonomy["Taxon_effectif_id"].eq(addition["Taxon_effectif_id"])
    ].copy()
    compatible_rows = branch_candidates.apply(
        lambda ref: compatible_with_regional_taxon(addition, ref), axis=1
    )
    branch_combinations = branch_candidates.loc[compatible_rows, TAXONOMY_RANKS].drop_duplicates()
    if len(branch_combinations) != 1:
        raise ValueError(
            f"Branche régionale absente ou ambiguë pour {new_class_name}: "
            f"{len(branch_combinations)} combinaison(s)."
        )
    reference_branch = branch_combinations.iloc[0]
    reference_row = branch_candidates.loc[compatible_rows].iloc[0]
    new_rows.append({
        "Classe": new_class_name,
        **{rank: reference_branch[rank] for rank in TAXONOMY_RANKS},
        "Commentaires": None,
        "Fréquent en Bassin AG": reference_row["Fréquent en Bassin AG"],
        "Niveau_precision": int(reference_row["Niveau_effectif"]),
        "Nb images original": 0,
        "Nb individus original": 0,
        "Provenance": "AquaMonitor",
        "Nb individus AquaMonitor": int(addition["Nb_individus_AquaMonitor"]),
        "Nb images AquaMonitor": int(addition["Nb_images_AquaMonitor"]),
    })

merged_reference = pd.concat([merged_reference, pd.DataFrame(new_rows)], ignore_index=True)
for column in ["Nb images original", "Nb individus original"]:
    merged_reference[column] = pd.to_numeric(merged_reference[column], errors="coerce").astype("Int64")
for column in ["Nb individus AquaMonitor", "Nb images AquaMonitor"]:
    merged_reference[column] = pd.to_numeric(merged_reference[column], errors="coerce").fillna(0).astype("int64")

# Hypothèse utilisateur : une image originale représente un individu original distinct.
merged_reference["Nb individus total"] = merged_reference["Nb individus original"].fillna(0) + merged_reference["Nb individus AquaMonitor"]
merged_reference["Nb images total"] = merged_reference["Nb images original"].fillna(0) + merged_reference["Nb images AquaMonitor"]
merged_reference = merged_reference[[
    "Classe", "REGNE", "EMBRANCHEMENT", "CLASSE", "ORDRE", "FAMILLE", "GENRE",
    "Commentaires", "Fréquent en Bassin AG", "Niveau_precision", "Provenance",
    "Nb images original", "Nb individus original", "Nb individus AquaMonitor", "Nb images AquaMonitor",
    "Nb individus total", "Nb images total",
]].sort_values("Classe", key=lambda values: values.astype("string").str.casefold(), na_position="last").reset_index(drop=True)

### Écrire et mettre en forme le référentiel enrichi

In [ ]:
merged_output = repo_root / "data" / "classes_presentes_dans_taxons_experiments_avec_aquamonitor_complet.xlsx"
with pd.ExcelWriter(merged_output, engine="openpyxl") as writer:
    merged_reference.to_excel(writer, sheet_name="Sheet1", index=False)

from openpyxl import load_workbook
from openpyxl.comments import Comment
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side

converted_workbook = load_workbook(merged_output)
converted_sheet = converted_workbook["Sheet1"]
converted_sheet.freeze_panes = "A2"
converted_sheet.auto_filter.ref = converted_sheet.dimensions
converted_sheet.sheet_view.showGridLines = False
for cell in converted_sheet[1]:
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = PatternFill("solid", fgColor="1F4E78")
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
widths = {
    "A": 76, "B": 12, "C": 19, "D": 17, "E": 20, "F": 23,
    "G": 20, "H": 36, "I": 23, "J": 18, "K": 24, "L": 20,
    "M": 22, "N": 25, "O": 22, "P": 20, "Q": 18,
}
for column, width in widths.items():
    converted_sheet.column_dimensions[column].width = width
thin_black = Side(style="thin", color="000000")
cell_border = Border(left=thin_black, right=thin_black, top=thin_black, bottom=thin_black)
class_colors = {
    "Original": "FCE4D6",
    "AquaMonitor": "D9EAF7",
    "Original + AquaMonitor": "E2F0D9",
}
headers = {cell.value: cell.column for cell in converted_sheet[1]}
for row in converted_sheet.iter_rows(min_row=1, max_row=converted_sheet.max_row, min_col=1, max_col=converted_sheet.max_column):
    for cell in row:
        cell.border = cell_border
for row_index in range(2, converted_sheet.max_row + 1):
    provenance = converted_sheet.cell(row_index, headers["Provenance"]).value
    class_cell = converted_sheet.cell(row_index, headers["Classe"])
    class_cell.fill = PatternFill("solid", fgColor=class_colors[provenance])
    converted_sheet.cell(row_index, headers["Commentaires"]).alignment = Alignment(wrap_text=True, vertical="top")
    for name in ["Nb images original", "Nb individus original", "Nb individus AquaMonitor", "Nb images AquaMonitor", "Nb individus total", "Nb images total"]:
        converted_sheet.cell(row_index, headers[name]).number_format = "#,##0"
    converted_sheet.row_dimensions[row_index].height = 30
converted_sheet.row_dimensions[1].height = 32
converted_sheet.cell(1, headers["Nb individus original"]).comment = Comment(
    "Hypothèse validée par l'utilisateur : chaque image originale représente un individu original distinct.",
    "Codex",
)
converted_workbook.save(merged_output)

assert len(merged_reference) == len(complete_reference) + len(new_additions)

### Vérifier que la hiérarchie n'a pas changé

In [ ]:
# Contrôle fondamental : aucune nouvelle combinaison taxonomique n'est créée.
def hierarchy_combinations(frame: pd.DataFrame) -> set[tuple]:
    """Return canonical REGNE-to-GENRE combinations for the invariance check."""
    return set(
        frame[TAXONOMY_RANKS]
        .fillna("<VIDE>")
        .astype(str)
        .itertuples(index=False, name=None)
    )

original_hierarchy_combinations = hierarchy_combinations(complete_reference)
merged_hierarchy_combinations = hierarchy_combinations(merged_reference)
assert merged_hierarchy_combinations == original_hierarchy_combinations
print("Combinaisons REGNE→GENRE avant/après :", len(original_hierarchy_combinations))

### Construire l'arbre final

In [ ]:
# Reconstruire l'arbre à partir de la référence complète fusionnée, et non plus
# uniquement à partir des lignes fréquentes dans le bassin.
full_tree_source = merged_reference.copy()
full_tree_source["Niveau_effectif"] = full_tree_source.apply(effective_level, axis=1)
full_tree_source["Taxon_effectif_id"] = full_tree_source.apply(effective_taxon_id, axis=1)
full_tree_source["CLASSE DATASET"] = full_tree_source["Classe"].fillna("— aucune classe dataset —")
full_tree_source["Images_originales"] = full_tree_source["Nb images original"].fillna(0).astype("int64")
full_tree_source["Individus_AquaMonitor"] = full_tree_source["Nb individus AquaMonitor"].astype("int64")
full_tree_source["NB INDIVIDUS"] = full_tree_source["Nb individus total"].astype("int64")
full_tree_source["FRÉQUENT BASSIN AG"] = full_tree_source["Fréquent en Bassin AG"].fillna("")

horizontal_tree = full_tree_source[[
    *TAXONOMY_RANKS, "CLASSE DATASET", "Images_originales",
    "Individus_AquaMonitor", "Provenance", "NB INDIVIDUS",
    "FRÉQUENT BASSIN AG", "Taxon_effectif_id",
]].sort_values([*TAXONOMY_RANKS, "CLASSE DATASET"], na_position="last").reset_index(drop=True)

assert hierarchy_combinations(horizontal_tree) == original_hierarchy_combinations
assert horizontal_tree["FRÉQUENT BASSIN AG"].eq("x").sum() == merged_reference["Fréquent en Bassin AG"].eq("x").sum()

full_level_tables = {}
for level_index, rank in enumerate(TAXONOMY_RANKS):
    path_columns = TAXONOMY_RANKS[: level_index + 1]
    full_level_tables[rank] = (
        horizontal_tree.groupby(path_columns, dropna=False)
        .agg(
            Nombre_taxons=("Taxon_effectif_id", "nunique"),
            Nombre_classes=("CLASSE DATASET", "nunique"),
            Images_originales=("Images_originales", "sum"),
            Individus_AquaMonitor=("Individus_AquaMonitor", "sum"),
            Nb_individus=("NB INDIVIDUS", "sum"),
        )
        .reset_index()
    )

full_tree_readme = pd.DataFrame({
    "Champ": [
        "Périmètre de l'arbre", "Lignes de données", "Lignes fréquentes bassin",
        "Lignes non fréquentes bassin", "Classes ajoutées AquaMonitor",
        "Combinaisons REGNE→GENRE", "Hypothèse individus originaux",
    ],
    "Valeur": [
        "Référence complète + classes AquaMonitor valides", len(horizontal_tree),
        int(horizontal_tree["FRÉQUENT BASSIN AG"].eq("x").sum()),
        int((~horizontal_tree["FRÉQUENT BASSIN AG"].eq("x")).sum()),
        int(merged_reference["Provenance"].eq("AquaMonitor").sum()),
        len(original_hierarchy_combinations), "1 image originale = 1 individu original",
    ],
})

### Exporter les résultats

In [ ]:
# Arbre visuel et vue filtrable : deux présentations des mêmes données.
export_filterable_tree(merged_reference, output_path)
print("Arbre complet régénéré :", output_path)
display(merged_reference)
print("Référence complète enrichie :", merged_output)

# Contrôles finaux indépendants de la taille du dataset.
assert len(merged_reference) == len(complete_reference) + len(new_rows)
assert len(horizontal_tree) == len(merged_reference)
assert merged_reference["Nb individus AquaMonitor"].sum() == matched_individuals["individual"].nunique()
assert merged_reference["Nb images AquaMonitor"].sum() == len(filtered_am_images)
assert hierarchy_combinations(merged_reference) == hierarchy_combinations(complete_reference)
print("Lignes originales :", len(complete_reference), "| classes ajoutées :", len(new_rows))
print("Individus retenus :", matched_individuals["individual"].nunique())

## 10. Résultats

À la fin du notebook, on obtient :

- la liste des individus retenus et exclus ;
- les classes retrouvées et ajoutées ;
- des graphiques simples ;
- un référentiel Excel enrichi ;
- un arbre Excel de la hiérarchie.

Les assertions servent de garde-fous. Si une règle importante n'est plus respectée,
le notebook s'arrête au lieu de produire silencieusement un mauvais fichier.